# 🤖 Tech Challenge Fase 3 — Fase 2: Fine-tuning com QLoRA

> **⚡ Otimizado para Colab Free (T4 16 GB):** QLoRA 4-bit + TinyLlama 1.1B

**Pipeline:**
```
1. Instalação de dependências (trl, peft, bitsandbytes, transformers)
2. Verificação da GPU
3. Carregamento do dataset Alpaca JSONL (saída do notebook 01)
4. Formatação do prompt no template Alpaca
5. Carregamento do modelo TinyLlama com quantização 4-bit
6. Configuração do LoRA (PEFT)
7. Treinamento supervisionado com SFTTrainer
8. Salvamento do adaptador LoRA
9. Inferência de teste e avaliação qualitativa
```

### Modelo base escolhido
| Parâmetro | Valor |
|-----------|-------|
| Modelo | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` |
| Quantização | 4-bit NF4 (QLoRA) |
| Técnica | LoRA (Low-Rank Adaptation) |
| VRAM estimada | ~4 GB |

> ⚠️ **Aviso:** Este assistente é para fins educacionais. Jamais substituir avaliação médica humana.

## ☁️ 0. Google Drive — Workspace Compartilhado
> Monta o Drive para que o dataset gerado pelo `01_dataset.ipynb` e o adaptador LoRA fiquem persistentes entre sessões e acessíveis por outros notebooks.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/tech-challenge-fase3'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f'✅ Drive montado. Pasta do projeto: {PROJECT_DIR}')
except ImportError:
    # Fora do Colab — usa o diretório local normalmente
    print('ℹ️  Ambiente local detectado. Usando diretório atual:', os.getcwd())

## 📦 1. Instalação de Dependências

In [ ]:
!pip install -q -U \
    "transformers>=4.46.0" \
    "datasets>=2.21.0" \
    "peft>=0.13.0" \
    "trl>=0.12.0" \
    "bitsandbytes>=0.44.0" \
    "accelerate>=0.34.2" \
    "fsspec>=2025.3.0" \
    scipy

print('✅ Dependências instaladas!')
print('⚠️  Se for a primeira execução: Runtime → Restart session → execute novamente a partir daqui.')

## 🖥️ 2. Verificação de GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')
else:
    print('⚠️  Sem GPU detectada. Ative: Runtime → Change runtime type → T4 GPU')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 📂 3. Carregamento do Dataset Alpaca JSONL
> Arquivo gerado pelo `01_dataset.ipynb`. Se estiver no Colab, faça upload de `data/dataset_medico.jsonl`.

In [ ]:
import os, json
import pandas as pd
from datasets import Dataset

JSONL_PATH = 'data/dataset_medico.jsonl'

# Se no Colab e arquivo não existe, permite upload manual
if not os.path.exists(JSONL_PATH):
    try:
        from google.colab import files
        print('📤 Faça upload do arquivo dataset_medico.jsonl:')
        uploaded = files.upload()
        os.makedirs('data', exist_ok=True)
        for fname, content in uploaded.items():
            with open(JSONL_PATH, 'wb') as f:
                f.write(content)
    except ImportError:
        raise FileNotFoundError(
            f'Arquivo não encontrado: {JSONL_PATH}\n'
            'Execute primeiro o notebook 01_dataset.ipynb.'
        )

records = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'✅ Dataset carregado: {len(df)} exemplos')
print(df[['instruction', 'input', 'output']].head(3))

## ✏️ 4. Formatação do Prompt (template Alpaca)

In [ ]:
ALPACA_TEMPLATE = (
    "Abaixo está uma instrução que descreve uma tarefa médica.\n"
    "Responda de forma precisa e segura, sempre indicando que a decisão final é do médico.\n\n"
    "### Instrução:\n{instruction}\n\n"
    "{input_block}"
    "### Resposta:\n{output}"
)

def formatar_prompt(exemplo: dict) -> dict:
    inp = exemplo.get('input', '')
    input_block = f'### Contexto:\n{inp}\n\n' if inp and inp.strip() else ''
    texto = ALPACA_TEMPLATE.format(
        instruction=exemplo['instruction'],
        input_block=input_block,
        output=exemplo['output']
    )
    return {'text': texto}

# Aplicar
dataset_hf = Dataset.from_pandas(df)
dataset_hf = dataset_hf.map(formatar_prompt, remove_columns=dataset_hf.column_names)

print(f'✅ {len(dataset_hf)} prompts formatados')
print('\n--- Exemplo ---')
print(dataset_hf[0]['text'][:600], '...')

## 🧠 5. Carregamento do Modelo Base com Quantização 4-bit (QLoRA)

**Modelo:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`  
**Por quê:** 1.1B parâmetros — cabe em T4 com 4-bit, treino rápido, bom para demonstração.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import bitsandbytes as bnb

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Verifica se quantização 4-bit está disponível (requer GPU + bitsandbytes com suporte CUDA)
USE_4BIT = torch.cuda.is_available() and bnb.cuda_setup.CUDASetup.get_instance().cuda_available

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    print('✅ Modo: QLoRA 4-bit (GPU disponível)')
else:
    bnb_config = None
    print('⚠️  bitsandbytes sem suporte CUDA — carregando em fp16 (sem quantização 4-bit)')
    print('   Verifique: Runtime → Change runtime type → T4 GPU')

print(f'⏳ Carregando tokenizer de {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'⏳ Carregando modelo...')
model_kwargs = dict(
    device_map='auto',
    trust_remote_code=True,
)
if USE_4BIT:
    model_kwargs['quantization_config'] = bnb_config
else:
    model_kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Modelo carregado — {total_params/1e6:.0f}M parâmetros')

## 🔧 6. Configuração do LoRA (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Preparar modelo para treino com k-bit
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,               # Rank — equilíbrio expressividade/memória
    lora_alpha=32,      # Escala = lora_alpha / r
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    # Camadas alvo do TinyLlama
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 🚀 7. Treinamento com SFTTrainer

> **Parâmetros conservadores** para Colab Free: batch 2 + grad_accum 4 = batch efetivo 8.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = 'outputs/tinyllama-medico'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_hf,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
)

print('🚀 Iniciando treinamento...')
train_result = trainer.train()
print('✅ Treinamento concluído!')
print(f'   Loss final : {train_result.training_loss:.4f}')
print(f'   Steps      : {train_result.global_step}')

## 📊 8. Curva de Treinamento

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps  = [e['step'] for e in log_history if 'loss' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

if steps:
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, marker='o', color='steelblue', linewidth=2)
    plt.title('Curva de Loss — Fine-tuning QLoRA (TinyLlama Médico)')
    plt.xlabel('Steps'); plt.ylabel('Loss')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=120)
    plt.show()
    print(f'✅ Curva salva em {OUTPUT_DIR}/loss_curve.png')
else:
    print('Sem dados de loss disponíveis.')

## 💾 9. Salvamento do Adaptador LoRA

In [ ]:
ADAPTER_DIR = 'outputs/tinyllama-medico-adapter'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f'✅ Adaptador LoRA salvo em: {ADAPTER_DIR}')
print('Arquivos:')
for f in os.listdir(ADAPTER_DIR):
    print(f'  {f}')

## 🧪 10. Inferência de Teste — Avaliação Qualitativa

> Comparar respostas do modelo ajustado com perguntas clínicas do dataset.

In [ ]:
from transformers import pipeline

# Recarregar em modo inferência (sem gradient checkpointing)
model.config.use_cache = True
model.eval()

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=True,
    repetition_penalty=1.1,
)

PERGUNTAS_TESTE = [
    {
        'instruction': 'Qual é o protocolo para manejo inicial de sepse?',
        'input': ''
    },
    {
        'instruction': 'Como conduzir dor torácica na emergência?',
        'input': ''
    },
    {
        'instruction': 'Quando indicar anticoagulação em fibrilação atrial?',
        'input': ''
    },
]

def gerar_resposta(instruction: str, inp: str = '') -> str:
    input_block = f'### Contexto:\n{inp}\n\n' if inp and inp.strip() else ''
    prompt = (
        'Abaixo está uma instrução que descreve uma tarefa médica.\n'
        'Responda de forma precisa e segura, sempre indicando que a decisão final é do médico.\n\n'
        f'### Instrução:\n{instruction}\n\n'
        f'{input_block}'
        '### Resposta:\n'
    )
    output = pipe(prompt)[0]['generated_text']
    # Extrai apenas a parte da resposta
    if '### Resposta:' in output:
        output = output.split('### Resposta:')[-1].strip()
    return output

for i, q in enumerate(PERGUNTAS_TESTE, 1):
    print(f'\n{'='*60}')
    print(f'❓ Pergunta {i}: {q["instruction"]}')
    resposta = gerar_resposta(q['instruction'], q.get('input', ''))
    print(f'💬 Resposta:\n{resposta}')
    print(f'{'='*60}')

## 📏 11. Avaliação Quantitativa — Perplexidade

> Perplexidade mede quão bem o modelo prediz o texto. Valores menores = melhor ajuste.

In [ ]:
import math

model.eval()
total_loss = 0.0
total_tokens = 0

# Avaliar em uma amostra para não sobrecarregar a VRAM
amostra = dataset_hf.select(range(min(20, len(dataset_hf))))

with torch.no_grad():
    for exemplo in amostra:
        enc = tokenizer(
            exemplo['text'],
            return_tensors='pt',
            truncation=True,
            max_length=512
        ).to(DEVICE)
        labels = enc['input_ids'].clone()
        out = model(**enc, labels=labels)
        n_tokens = labels.numel()
        total_loss   += out.loss.item() * n_tokens
        total_tokens += n_tokens

avg_loss    = total_loss / total_tokens
perplexidade = math.exp(avg_loss)

print(f'📊 Métricas de Avaliação (amostra de {len(amostra)} exemplos)')
print(f'   Loss médio   : {avg_loss:.4f}')
print(f'   Perplexidade : {perplexidade:.2f}')
print()
print('Interpretação:')
if perplexidade < 5:
    print('  ✅ Excelente — modelo bem ajustado ao domínio médico')
elif perplexidade < 15:
    print('  ✅ Bom — modelo aprendeu os padrões do domínio')
elif perplexidade < 40:
    print('  ⚠️  Razoável — mais épocas ou dados podem ajudar')
else:
    print('  ❌ Alto — verificar dados ou hiperparâmetros')

## ☁️ 12. Download dos Artefatos (Colab)

In [ ]:
import shutil

# Zipar o adaptador
zip_path = 'tinyllama-medico-adapter.zip'
shutil.make_archive('tinyllama-medico-adapter', 'zip', ADAPTER_DIR)
print(f'✅ Adaptador compactado: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
    files.download(f'{OUTPUT_DIR}/loss_curve.png')
    print('📥 Downloads iniciados!')
except ImportError:
    print(f'(Fora do Colab) Artefatos em: {ADAPTER_DIR} e {OUTPUT_DIR}/')

## ✅ Resumo da Fase 2

| Item | Status |
|------|--------|
| Modelo base | TinyLlama 1.1B |
| Quantização | 4-bit NF4 (QLoRA) |
| Adaptador LoRA | `outputs/tinyllama-medico-adapter/` |
| Curva de loss | `outputs/tinyllama-medico/loss_curve.png` |
| Perplexidade | calculada acima |

### 🚀 Próximo passo: Fase 3 — Assistente Médico com LangChain

O adaptador salvo em `outputs/tinyllama-medico-adapter/` será carregado no `03_langchain.ipynb` para construir o assistente com:
- RAG (Retrieval-Augmented Generation) com prontuários
- Chains de consulta contextualizada
- Logging e auditoria das respostas
- Fluxos LangGraph automatizados